# Scope Drift Analysis Pipeline

1. **cwts_export.py** — Fetch citation network, run CWTS clustering, upload to BigQuery
2. **label_clusters** — Generate GPT labels for clusters
3. **build_unified_dashboard.py** — Create combined dashboard from BigQuery data

In [4]:
from datetime import datetime

# Generate a new timestamp for this run, or use an existing one
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# run_timestamp = "20260714_151745"  # Uncomment to reuse existing run

# Config
START_YEAR = "2023"
END_YEAR = "2026"
NETWORK_MODE = "full"  # Options: ego, full, global
CLUSTER_LEVEL = "macro"  # Options: micro, meso, macro

# Journals to analyze (comma-separated, must match journal names in BigQuery)
# Leave empty to use defaults (Immunology, Public Health, Medicine, Oncology, Psychology)
JOURNALS = ",".join(
    [
        "Frontiers in Neurorobotics",
        "Frontiers in Earth Science",
        "Frontiers in Surgery",
        "Frontiers in Aging Neuroscience",
        "Frontiers in Environmental Science",
        "Frontiers in Chemistry",
        "Frontiers in Materials",
        "Frontiers in Robotics and AI",
    ]
)

print(f"run_timestamp: {run_timestamp}")
print(f"JOURNALS: {JOURNALS}")

run_timestamp: 20260728_134519
JOURNALS: Frontiers in Neurorobotics,Frontiers in Earth Science,Frontiers in Surgery,Frontiers in Aging Neuroscience,Frontiers in Environmental Science,Frontiers in Chemistry,Frontiers in Materials,Frontiers in Robotics and AI


In [ ]:
# Step 1: Run cwts_export.py
import subprocess, sys, os

env = os.environ.copy()
env["START_YEAR"] = START_YEAR
env["END_YEAR"] = END_YEAR
env["NETWORK_MODE"] = NETWORK_MODE
env["RUN_TIMESTAMP"] = run_timestamp

print(f"Running cwts_export.py with run_timestamp={run_timestamp}...")
result = subprocess.run(
    [sys.executable, "cwts_export.py"], capture_output=True, text=True, env=env
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

In [1]:
# timestamp = "20260710_144411" # 2020-2026 full run
# timestamp = "20260618_130306"  # 2023-2026 full run
timestamp = "20260714_151745"  # tes run with new journals
timestamp = "20260721_122750"  # test journals in eu
# timestamp = "20260722_222014"# full run test journals changed resolution -- to few clsters this time 10ish clusters
# timestamp = "20260723_150948"  # full run diff resolution 20ish clusters

In [2]:
# Step 2: Generate GPT cluster labels
from src import label_clusters

label_clusters.main(timestamp)

C:\Users\sophie.wilson\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Loading from BigQuery (run_timestamp=20260721_122750)...
  Classification: ocean-tech-adv-analytics-c-tfs.raw_citation_network_data.classification_raw_20260721_122750
  Pub metadata:   ocean-tech-adv-analytics-c-tfs.raw_citation_network_data.pub_metadata_raw_20260721_122750
  Cit links:      ocean-tech-adv-analytics-c-tfs.raw_citation_network_data.cit_links_raw_20260721_122750

Processing macro level...
  Fetching up to 250 titles per cluster...
Downloading: 100%|██████████|
  Loaded 12,500 titles across 50 clusters

--- Pass 1: Labelling macro (50 clusters) ---
  [1/50] cluster 0 (1,451,756 papers) → Electrochemical Materials
  [2/50] cluster 1 (1,182,469 papers) → Cancer Immunotherapy
  [3/50] cluster 2 (649,185 papers) → Environmental Monitoring
  [4/50] cluster 3 (597,219 papers) → Sustainable Development
  [5/50] cluster 4 (586,660 papers) → Neurodegenerative Diseases
  [6/50] cluster 5 (486,618 papers) → Gut Health Research
  [7/50] cluster 6 (451,253 papers) → Materials Science


In [ ]:
# # Step 3: Generate dashboard
import subprocess, sys, os

env = os.environ.copy()
env["RUN_TIMESTAMP"] = timestamp
env["CLUSTER_LEVEL"] = "macro"
if JOURNALS:
    env["JOURNALS"] = JOURNALS

# Borderline (amber): LLM judges non-primary communities as borderline vs hard OOS.
# Replaces layout-distance rescue (keep SCOPE_DISTANCE_ENABLED=0 unless debugging).
env["SCOPE_LLM_BORDERLINE_ENABLED"] = "1"
env["SCOPE_LLM_BORDERLINE_MODEL"] = "gpt-4o-mini"
env["SCOPE_LLM_BORDERLINE_PROMPT_VERSION"] = "v2"
env["SCOPE_DISTANCE_ENABLED"] = "0"
# Title hard-negatives + paper LLM demotion inside risky primary communities
env["SCOPE_HARD_NEGATIVES_ENABLED"] = "1"
env["SCOPE_PAPER_LLM_ENABLED"] = "1"
env["SCOPE_PAPER_LLM_MODEL"] = "gpt-4o-mini"
env["LAYOUT_SEED"] = "42"  # pin FR layout so scatter geometry is reproducible

result = subprocess.run(
    [sys.executable, "src/build_unified_dashboard.py"],
    capture_output=True,
    text=True,
    env=env,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode == 0:
    print("\nDashboard ready: output/combined_dashboard.html")

In [ ]:
# Step 4: Build GT network map (after Step 3 dashboards)
# Overlays manual_scope_check_truth.ods on the current scope_dashboard.html layout.
import subprocess, sys, os

env = os.environ.copy()
env["RUN_TIMESTAMP"] = timestamp  # same run as Step 3
env["CLUSTER_LEVEL"] = "macro"
if JOURNALS:
    env["JOURNALS"] = JOURNALS

result = subprocess.run(
    [sys.executable, "scripts/build_gt_network_map.py"],
    capture_output=True,
    text=True,
    env=env,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode == 0:
    print("\nGT map ready: output/gt_network_map.html")
else:
    raise SystemExit(result.returncode)

---
# Old Cells (Reference)

In [ ]:
# import subprocess, sys, os

# """
# This output is:
#  1. pubs.txt - Paper list for CWTS tool
#     Columns: int_id  paper identifier for cwts, core_pub (always 1 as not using core feature, but it has to be in)

#  2. cit_links.txt - Citation network edges
#     Columns:
#     int_id1 - citing paper,
#     int_id2 - cited paper,
#     weight - citation strength 0-2 higher= stronger
#     Note: Each edge appears twice (A→B and B→A) for undirected format
#         paper 5 cites Paper 12  →  row: 5, 12, 0.85
#         Paper 12 cites Paper 5  →  row: 12, 5, 0.85  (same edge, reversed)

#  3. pub_metadata.txt - Paper details lookup table
#     Columns: int_id, pub_id, is_frontiers, journal, date, title
#     - int_id: sequential CWTS ID (joins to classification.txt)
#     - pub_id: airak PublicationId (joins to BigQuery tables)
#  JOIN KEY: int_id links all files together, this is cwts identifier
# """
# print("ere")
# env = os.environ.copy()
# print("ere")
# env["START_YEAR"] = "2023"
# env["END_YEAR"] = "2026"
# env["NETWORK_MODE"] = "full"
# env["RUN_TIMESTAMP"] = run_timestamp  # Must be uppercase to match cwts_export.py
# print("ere")
# result = subprocess.run(
#     [sys.executable, "cwts_export.py"], capture_output=True, text=True, env=env
# )
# print("ere")
# print(result.stdout)
# print(result.stderr)
# print("last")

In [ ]:
# import subprocess
# import datetime
# import os
# import pandas as pd

# # --- Parameters ---
# params = {
#     "largest_component_only": "true",
#     "iterations": "100",
#     "micro_resolution": "5e-4",
#     "micro_min_cluster_size": "1000",
#     "meso_resolution": "5e-6",
#     "meso_min_cluster_size": "5000",
#     "macro_resolution": "1e-6",
#     "macro_min_cluster_size": "20000",
# }


# input_files = {
#     "pubs": "cwts_output/pubs.txt",
#     "cit_links": "cwts_output/cit_links.txt",
#     "output": "cwts_output/classification.txt",
#     "jar": "publicationclassification.jar",
# }


# result = subprocess.run(
#     [
#         "java",
#         "-cp",
#         input_files["jar"],
#         "nl.cwts.publicationclassification.run.PublicationClassificationCreator",
#         input_files["pubs"],
#         input_files["cit_links"],
#         input_files["output"],
#         params["largest_component_only"],
#         params["iterations"],
#         params["micro_resolution"],
#         params["micro_min_cluster_size"],
#         params["meso_resolution"],
#         params["meso_min_cluster_size"],
#         params["macro_resolution"],
#         params["macro_min_cluster_size"],
#     ],
#     capture_output=True,
#     text=True,
# )

# # --- Log ---
# os.makedirs("logs", exist_ok=True)
# log_path = f"logs/cwts_run_{run_timestamp}.log"

# with open(log_path, "w") as f:
#     f.write(f"CWTS Publication Classification Run\n")
#     f.write(f"{'='*50}\n")
#     f.write(f"Timestamp : {run_timestamp}\n\n")

#     f.write(f"Input Files\n{'-'*30}\n")
#     for k, v in input_files.items():
#         f.write(f"  {k:<20}: {v}\n")

#     f.write(f"\nParameters\n{'-'*30}\n")
#     for k, v in params.items():
#         f.write(f"  {k:<26}: {v}\n")

#     f.write(f"\nReturn Code: {result.returncode}\n")

#     f.write(f"\nSTDOUT\n{'-'*30}\n")
#     f.write(result.stdout or "(empty)\n")

#     f.write(f"\nSTDERR\n{'-'*30}\n")
#     f.write(result.stderr or "(empty)\n")

# print(f"Log written to: {log_path}")
# print(result.stdout)
# if result.stderr:
#     print(result.stderr)

# # Load classification.txt
# classification = pd.read_csv(
#     "cwts_output/classification.txt",
#     sep="\t",
#     header=None,
#     names=["int_id", "micro", "meso", "macro"],
# )

# # Upload to BigQuery
# BQ_DEST_PROJECT = "ocean-tech-adv-analytics-c-tfs"
# BQ_DEST_DATASET = "scope_drift_raw"


# # classification.to_gbq(
# #     f"{dataset}.classification_raw_{run_timestamp}",
# #     project_id=project,
# #     if_exists="replace",
# # )

# # Upload to BigQuery

# classification.to_gbq(
#     f"{BQ_DEST_DATASET}.classification_raw_{run_timestamp}",
#     project_id=BQ_DEST_PROJECT,
#     if_exists="replace",
# )
# print(f"  → BigQuery: {BQ_DEST_DATASET}.classification_raw_{run_timestamp}")

In [ ]:
# import pandas as pd


# df = pd.read_csv(
#     "cwts_output/classification.txt",
#     sep="\t",
#     header=None,
#     names=["pub_no", "micro", "meso", "macro"],
# )


# print(f"Total classified: {len(df):,}")


# for level in ["micro", "meso", "macro"]:

#     vc = df[level].value_counts()

#     print(f"\n{level.upper()}: {len(vc):,} clusters")

#     print(f"  Largest : {vc.iloc[0]:,} ({vc.iloc[0]/len(df)*100:.1f}%)")

#     print(f"  Smallest: {vc.iloc[-1]:,}")

#     print(f"  Median  : {vc.median():.0f}")

### Labelling with GPT

In [ ]:
# timestamp = "20260710_144411" # 2020-2026 full run
# timestamp = "20260618_130306"  # 2023-2026 full run
# timestamp = "20260714_151745"  # tes run with new journals

In [ ]:
# import label_clusters

# # Run the script (reads from BigQuery using run_timestamp)
# label_clusters.main(timestamp)

## Labelling with taxonomy 
- this is not as good as raw GPT so for now im going to drop it and ill ask toby what to do later

In [ ]:
# import taxonomy_naming

# # Run taxonomy naming (reads from BigQuery using run_timestamp)
# df_out = taxonomy_naming.main(timestamp)

### looking at scope

In [ ]:
# import importlib
# import journal_scope
# from pathlib import Path
# import os

# # Override config variables
# journal_scope.SCOPE_LEVEL = "macro"
# journal_scope.SCOPE_THRESHOLD = 0.80
# journal_scope.MIN_PAPERS = 50
# journal_scope.USE_GPT = False
# journal_scope.OUTPUT_DIR = Path("cwts_output")
# # Data source - set on the module, not as local variables
# journal_scope.DATA_SOURCE = "bigquery"
# journal_scope.RUN_TIMESTAMP = timestamp
# journal_scope.TARGET_JOURNALS = [
#     "Frontiers in Immunology",
#     "Frontiers in Public Health",
#     "Frontiers in Medicine",
#     "Frontiers in Oncology",
#     "Frontiers in Psychology",
# ]
# #
# journal_scope.main()

### Generate Dashboard

In [ ]:
# Analyze specific papers for scope status
# Uses the latest run data to determine if papers are in/out of scope
# Ground truth: manual_scope_class from review_packs_scope_checked.xlsx

import re
import pandas as pd
from google.cloud import bigquery
import json
from pathlib import Path

# Config - use the timestamp from the latest run
RUN_TIMESTAMP = timestamp  # From cell 1
BQ_PROJECT = "ocean-tech-adv-analytics-c-tfs"  # Same as cwts_export.py BQ_DEST_PROJECT
BQ_DATASET = "raw_citation_network_data"
PRIMARY_COVERAGE = 0.8  # 80% coverage for primary clusters
CLUSTER_LEVEL = "macro"

GROUND_TRUTH_PATH = r"C:\Users\sophie.wilson\Downloads\review_packs_scope_checked.xlsx"

# Load CSV to analyze
csv_path = r"C:\Users\sophie.wilson\Downloads\scope_drift_matches.csv"
papers_df = pd.read_csv(csv_path)
print(f"Loaded {len(papers_df)} papers to analyze")


def norm_title(s):
    if pd.isna(s):
        return ""
    t = str(s).lower()
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def norm_gt_label(x):
    """Map manual labels to In Scope / Out of Scope."""
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()
    if "out of scope" in s or s in ("oos", "0", "false", "no") or s.startswith("oos"):
        return "Out of Scope"
    if "in scope" in s or s in ("in", "yes", "1", "true"):
        return "In Scope"
    return str(x)


# Load ground truth (manual review packs)
gt = pd.read_excel(GROUND_TRUTH_PATH, sheet_name="unique_articles")
gt["ground_truth_scope"] = gt["manual_scope_class"].map(norm_gt_label)
gt["title_norm"] = gt["title"].map(norm_title)
gt["article_id_original"] = pd.to_numeric(gt["article_id_original"], errors="coerce")
print(f"Loaded {len(gt)} ground-truth rows from {GROUND_TRUTH_PATH}")

# Prefer article_id_original match; fall back to normalized title
papers_df["article_id_original"] = pd.to_numeric(
    papers_df.get("article_id_original"), errors="coerce"
)
title_col = "article_name" if "article_name" in papers_df.columns else "title"
papers_df["title_norm"] = papers_df[title_col].map(norm_title)

gt_by_id = (
    gt.dropna(subset=["article_id_original"])
    .drop_duplicates("article_id_original")[
        [
            "article_id_original",
            "ground_truth_scope",
            "manual_scope_score",
            "source_file",
        ]
    ]
    .rename(
        columns={
            "manual_scope_score": "ground_truth_score",
            "source_file": "gt_source_file",
        }
    )
)
gt_by_title = (
    gt.dropna(subset=["title_norm"])
    .drop_duplicates("title_norm")[
        ["title_norm", "ground_truth_scope", "manual_scope_score", "source_file"]
    ]
    .rename(
        columns={
            "manual_scope_score": "ground_truth_score",
            "source_file": "gt_source_file",
        }
    )
)

papers_df = papers_df.merge(gt_by_id, on="article_id_original", how="left")
missing_gt = papers_df["ground_truth_scope"].isna()
if missing_gt.any():
    title_merge = papers_df.loc[missing_gt, ["title_norm"]].merge(
        gt_by_title, on="title_norm", how="left"
    )
    papers_df.loc[missing_gt, "ground_truth_scope"] = title_merge[
        "ground_truth_scope"
    ].values
    papers_df.loc[missing_gt, "ground_truth_score"] = title_merge[
        "ground_truth_score"
    ].values
    papers_df.loc[missing_gt, "gt_source_file"] = title_merge["gt_source_file"].values

n_gt = papers_df["ground_truth_scope"].notna().sum()
print(f"Matched ground truth for {n_gt}/{len(papers_df)} papers")

# Load full dataset from BigQuery to compute primary clusters
client = bigquery.Client(project=BQ_PROJECT, location="EU")  # EU location required
tbl_classif = f"{BQ_PROJECT}.{BQ_DATASET}.classification_raw_{RUN_TIMESTAMP}"
tbl_pub_meta = f"{BQ_PROJECT}.{BQ_DATASET}.pub_metadata_raw_{RUN_TIMESTAMP}"

query = f"""
SELECT c.int_id, c.macro, m.journal
FROM `{tbl_classif}` c
JOIN `{tbl_pub_meta}` m ON c.int_id = m.int_id
WHERE m.is_frontiers = 1
"""
full_df = client.query(query).to_dataframe()
print(f"Loaded {len(full_df):,} Frontiers papers from BigQuery")


# Compute primary clusters for each journal
def get_primary_clusters(df, journal, coverage=0.8):
    jdf = df[df["journal"] == journal]
    if jdf.empty:
        return set()
    cluster_counts = jdf["macro"].value_counts()
    total = cluster_counts.sum()
    cumsum = 0
    primary = set()
    for cluster, count in cluster_counts.items():
        primary.add(cluster)
        cumsum += count
        if cumsum / total >= coverage:
            break
    return primary


# Get unique journals from our papers
journals_in_csv = papers_df["journal"].unique()
primary_clusters = {}
for j in journals_in_csv:
    # Match journal name format (Frontiers in X)
    full_name = "Frontiers in " + j.replace("frontiers in ", "").title()
    primary_clusters[j] = get_primary_clusters(full_df, full_name)
    print(
        f"{full_name}: {len(primary_clusters[j])} primary clusters: {sorted(primary_clusters[j])}"
    )

# Load GPT labels
labels_path = Path("cwts_output/gpt_labels_macro.json")
gpt_labels = {}
if labels_path.exists():
    with open(labels_path) as f:
        labels_data = json.load(f)
        for item in labels_data:
            gpt_labels[item["cluster_id"]] = item.get(
                "short_label", f"Cluster {item['cluster_id']}"
            )
    print(f"Loaded {len(gpt_labels)} GPT labels")


# Determine scope status for each paper
def get_scope_status(row):
    journal_key = row["journal"]
    cluster = row["macro_community"]
    primary = primary_clusters.get(journal_key, set())
    return "In Scope" if cluster in primary else "Out of Scope"


def get_community_label(cluster_id):
    return gpt_labels.get(cluster_id, f"Cluster {cluster_id}")


papers_df["scope_status"] = papers_df.apply(get_scope_status, axis=1)
papers_df["community_label"] = papers_df["macro_community"].apply(get_community_label)
papers_df["agreement"] = papers_df.apply(
    lambda r: (
        pd.NA
        if pd.isna(r["ground_truth_scope"])
        else ("Agree" if r["scope_status"] == r["ground_truth_scope"] else "Disagree")
    ),
    axis=1,
)

# Display results
print("\n" + "=" * 80)
print("SCOPE ANALYSIS RESULTS (model vs ground truth)")
print("=" * 80)

for journal in papers_df["journal"].unique():
    jdf = papers_df[papers_df["journal"] == journal]
    in_scope = (jdf["scope_status"] == "In Scope").sum()
    out_scope = (jdf["scope_status"] == "Out of Scope").sum()
    gt_known = jdf["ground_truth_scope"].notna().sum()
    agree = (jdf["agreement"] == "Agree").sum()
    print(f"\n{journal.upper()}")
    print(
        f"  Model — In scope: {in_scope}, Out of scope: {out_scope} | "
        f"GT matched: {gt_known} | Agree: {agree}/{gt_known if gt_known else 0}"
    )
    print("-" * 70)
    for _, row in jdf.iterrows():
        status_marker = "[OK]" if row["scope_status"] == "In Scope" else "[OOS]"
        gt = row["ground_truth_scope"] if pd.notna(row["ground_truth_scope"]) else "—"
        agr = row["agreement"] if pd.notna(row["agreement"]) else "—"
        title = (
            row[title_col][:60] + "..."
            if len(str(row[title_col])) > 60
            else row[title_col]
        )
        print(
            f"  {status_marker} model={row['scope_status']:12} | GT={str(gt):12} | {agr:8} | "
            f"C{row['macro_community']:2} {str(row['community_label'])[:24]:24} | {title}"
        )

# Overall agreement
comparable = papers_df[papers_df["agreement"].isin(["Agree", "Disagree"])]
if len(comparable):
    print("\n" + "=" * 80)
    print("GROUND TRUTH AGREEMENT")
    print("=" * 80)
    print(f"Comparable papers: {len(comparable)}")
    print(f"Agreement: {(comparable['agreement'] == 'Agree').mean():.1%}")
    print(
        pd.crosstab(
            comparable["ground_truth_scope"], comparable["scope_status"], margins=True
        )
    )

# Save enriched CSV
output_path = (
    rf"C:\Users\sophie.wilson\Downloads\scope_drift_matches_analyzed{timestamp}.csv"
)
papers_df.to_csv(output_path, index=False)
print(f"\n\nSaved enriched results to: {output_path}")

Loaded 83 papers to analyze
Loaded 158 ground-truth rows from C:\Users\sophie.wilson\Downloads\review_packs_scope_checked.xlsx
Matched ground truth for 83/83 papers


c:\Users\sophie.wilson\AppData\Local\miniconda3\envs\scope_drift\lib\site-packages\google\cloud\bigquery\table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded 38,222 Frontiers papers from BigQuery
Frontiers in Aging Neuroscience: 3 primary clusters: [1, 4, 14]
Frontiers in Earth Science: 3 primary clusters: [2, 9, 13]
Frontiers in Environmental Science: 3 primary clusters: [2, 3, 7]
Frontiers in Neurorobotics: 4 primary clusters: [4, 8, 15, 27]
Frontiers in Surgery: 10 primary clusters: [1, 4, 14, 21, 22, 24, 25, 26, 34, 36]

SCOPE ANALYSIS RESULTS (model vs ground truth)

FRONTIERS IN AGING NEUROSCIENCE
  Model — In scope: 16, Out of scope: 3 | GT matched: 19 | Agree: 10/19
----------------------------------------------------------------------
  [OK] model=In Scope     | GT=Out of Scope | Disagree | C 4 Cluster 4                | A cross-database bibliometric analysis of rapid eye movement...
  [OK] model=In Scope     | GT=In Scope     | Agree    | C 4 Cluster 4                | Adolescent Binge Alcohol Exposure Accelerates Alzheimer's di...
  [OK] model=In Scope     | GT=Out of Scope | Disagree | C 4 Cluster 4                | Astro

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\sophie.wilson\\Downloads\\scope_drift_matches_analyzed20260721_122750.csv'

In [ ]:
# import subprocess
# import os

# env = os.environ.copy()
# env["RUN_TIMESTAMP"] = timestamp
# env["CLUSTER_LEVEL"] = "macro"  # optional
# # env["JOURNALS"] = "Frontiers in Immunology,Frontiers in Oncology"  # optional override
# env["BUILD_ALL"] = "1"  # <-- Add this to regenerate dashboards
# result = subprocess.run(
#     ["python", "scripts/build_unified_dashboard.py"],
#     capture_output=True,
#     text=True,
#     env=env,
# )
# # Checking for errors as this is claude generated
# print("Return code:", result.returncode)
# print("STDOUT:", result.stdout)
# print("STDERR:", result.stderr)

In [ ]:
# import subprocess
# import sys
# import os

# # --- Config ---
# CLUSTER_LEVEL = "macro"  # Options: micro, meso, macro

# # Build the scope dashboard from local CWTS files
# # Uses: cwts_output/classification.txt, cwts_output/pub_metadata.txt, cwts_output/cit_links.txt
# # Outputs: output/scope_dashboard.html
# # Metadata pulled from BigQuery using run_timestamp
# print(timestamp)
# env = os.environ.copy()
# # env["CLUSTER_LEVEL"] = CLUSTER_LEVEL
# env["RUN_TIMESTAMP"] = timestamp  # Pass timestamp to fetch metadata from BigQuery
# env["DATA_SOURCE"] = "bigquery"

# result = subprocess.run(
#     [sys.executable, "scripts/build_dashboard_from_cwts.py"],
#     capture_output=True,
#     text=True,
#     env=env,
# )

# print(result.stdout)
# if result.returncode != 0:
#     print("ERRORS:")
#     print(result.stderr)
# else:
#     print(f"\nDashboard ready: output/scope_dashboard.html")

In [ ]:
# import subprocess
# import sys
# import os

# # --- Config ---
# CLUSTER_LEVEL = "macro"  # Options: micro, meso, macro

# # Build the drift dashboard (JSD trends, heatmap, entropy changes)
# # Compares current cluster distribution vs baseline (2018-2020)
# # Outputs: output/drift_dashboard.html

# env = os.environ.copy()
# env["CLUSTER_LEVEL"] = CLUSTER_LEVEL
# env["RUN_TIMESTAMP"] = timestamp  # Pass timestamp to fetch metadata from BigQuery

# result = subprocess.run(
#     [sys.executable, "scripts/build_drift_dashboard_from_cwts.py"],
#     capture_output=True,
#     text=True,
#     env=env,
# )

# print(result.stdout)
# if result.returncode != 0:
#     print("ERRORS:")
#     print(result.stderr)
# else:
#     print(f"\nDashboard ready: output/drift_dashboard.html")

In [ ]:
# import subprocess
# import sys
# import os

# # --- Config ---
# CLUSTER_LEVEL = "macro"  # Options: micro, meso, macro

# # Build the cluster bubble dashboard
# # Shows clusters as bubbles positioned by citation relationships
# # Outputs: output/cluster_bubbles.html

# env = os.environ.copy()
# env["CLUSTER_LEVEL"] = CLUSTER_LEVEL

# result = subprocess.run(
#     [sys.executable, "scripts/build_cluster_bubbles_from_cwts.py"],
#     capture_output=True,
#     text=True,
#     env=env,
# )

# print(result.stdout)
# if result.returncode != 0:
#     print("ERRORS:")
#     print(result.stderr)
# else:
#     print(f"\nDashboard ready: output/cluster_bubbles.html")

In [ ]:
# import subprocess
# import sys

# # Build the clusters hierarchy dashboard
# # Shows macro/meso/micro clusters with GPT labels
# # Outputs: output/clusters.html

# result = subprocess.run(
#     [sys.executable, "scripts/build_clusters_from_cwts.py"],
#     capture_output=True,
#     text=True,
# )

# print(result.stdout)
# if result.returncode != 0:
#     print("ERRORS:")
#     print(result.stderr)
# else:
#     print(f"\nDashboard ready: output/clusters.html")